In [1]:
from dataclasses import dataclass
from faster_whisper.transcribe import Segment

@dataclass
class TranscriptionResult:
    text: str
    language: str
    language_probability: float
    segments: list[Segment]
    audio_duration_s: float
    inference_time_s: float

    @property
    def realtime_factor(self) -> float:
        return self.audio_duration_s / self.inference_time_s

In [2]:
from faster_whisper import WhisperModel

device = "cpu"
compute_type = "float32"

# Audio config
SAMPLE_RATE = 16_000
CHANNELS = 1
DTYPE = "float32"

# Chunking config
CHUNK_DURATION_S = 3       # transcribe every N seconds
OVERLAP_DURATION_S = 0.5   # overlap between chunks to avoid cutting words mid-sentence
model = WhisperModel("large-v3-turbo", device=device, compute_type=compute_type)

In [3]:
import time
import queue
import tempfile
import threading
import soundfile as sf
import sounddevice as sd
import numpy as np
from collections import deque

def _audio_callback(indata: np.ndarray, frames: int, time_info, status, q: queue.Queue):
    if status:
        print(f"[audio] {status}")
    q.put(indata.copy().flatten())  # always 1D

def stream_transcribe(
    model: WhisperModel, 
    stop_event: threading.Event, 
    language: str | None = None, 
    chunk_s: float = CHUNK_DURATION_S, 
    overlap_s: float = OVERLAP_DURATION_S,
    output=None,
    device: int | None = None,
):
    def emit(msg: str):
        if output is not None:
            with output:
                print(msg)
        else:
            print(msg)

    device_idx = device if device is not None else sd.default.device[0]
    device_name = sd.query_devices(device_idx)["name"]
    emit(f"[mic] using device #{device_idx}: {device_name}")

    audio_queue: queue.Queue[np.ndarray] = queue.Queue()
    chunk_samples = int(chunk_s * SAMPLE_RATE)
    overlap_samples = int(overlap_s * SAMPLE_RATE)

    buffer: deque[np.ndarray] = deque()
    buffer_len = 0
    start_time = time.time()
    chunk_idx = 0

    def transcription_loop():
        nonlocal buffer, buffer_len, chunk_idx

        while not stop_event.is_set():
            try:
                while True:
                    chunk = audio_queue.get_nowait()  # already 1D
                    buffer.append(chunk)
                    buffer_len += len(chunk)
            except queue.Empty:
                pass

            if buffer_len < chunk_samples:
                time.sleep(0.05)
                continue

            audio_np = np.concatenate(list(buffer))  # all 1D, safe

            overlap_audio = audio_np[-overlap_samples:] if overlap_samples > 0 else np.array([])
            buffer = deque([overlap_audio]) if len(overlap_audio) > 0 else deque()
            buffer_len = len(overlap_audio)

            chunk_idx += 1
            elapsed = time.time() - start_time

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as f:
                sf.write(f.name, audio_np[:chunk_samples], SAMPLE_RATE)
                segments_gen, info = model.transcribe(
                    f.name,
                    language=language,
                    beam_size=5,
                    vad_filter=True,
                    vad_parameters=dict(
                        min_silence_duration_ms=200,  # short chunk: don't wait for long silence
                        speech_pad_ms=200,
                        threshold=0.3,
                    ),
                    word_timestamps=False,
                )
                segments = list(segments_gen)
                text = " ".join(seg.text.strip() for seg in segments)

            emit(f"[chunk {chunk_idx:03d} | {elapsed:6.1f}s | {info.language}] {text if text.strip() else '(silence)'}")

    t = threading.Thread(target=transcription_loop, daemon=True)
    t.start()

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=CHANNELS,
        dtype=DTYPE,
        blocksize=int(0.1 * SAMPLE_RATE),
        device=device_idx,
        callback=lambda indata, frames, time_info, status: _audio_callback(
            indata, frames, time_info, status, audio_queue
        ),
    ):
        while not stop_event.is_set():
            time.sleep(0.1)

    t.join(timeout=5)

In [4]:
import sounddevice as sd

print(sd.query_devices())

DEFAULT_MIC_NAME = "MacBook Pro Microphone"

def _find_device_by_name(name: str) -> int:
    for i, d in enumerate(sd.query_devices()):
        if d["name"] == name and d["max_input_channels"] > 0:
            return i
    print(f"[mic] '{name}' not found, falling back to system default")
    return sd.default.device[0]

MIC_DEVICE = _find_device_by_name(DEFAULT_MIC_NAME)

stop_event = threading.Event()
try:
    stream_transcribe(model=model, stop_event=stop_event, chunk_s=3.0, device=MIC_DEVICE)
except KeyboardInterrupt:
    stop_event.set()

  0 Meeko Microphone, Core Audio (1 in, 0 out)
> 1 Studio Display Microphone, Core Audio (1 in, 0 out)
< 2 Studio Display Speakers, Core Audio (0 in, 8 out)
  3 MacBook Pro Microphone, Core Audio (1 in, 0 out)
  4 MacBook Pro Speakers, Core Audio (0 in, 2 out)
[mic] using device #3: MacBook Pro Microphone
[chunk 001 |    3.1s | en] We'll be right back.
[chunk 002 |    7.1s | en] temporarily replaced me
[chunk 003 |   11.0s | en] So a substitute is something that...
[chunk 004 |   14.8s | en] So, if there doesn't know me...
[chunk 005 |   18.7s | en] Well, the menu explains everything.
[chunk 006 |   22.6s | en] for your hamburger patties. They're all plant-based.
[chunk 007 |   26.5s | en] I think I will get that cowboy bird.
[chunk 008 |   30.4s | en] What about you up there? Well, it might be a little adventure.


[chunk 009 |   34.5s | en] that hummus wrap. Abdul, hummus.
